In [ ]:
!pip install albumentations

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler
import random
import copy

# Data augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4
print(f"Device: {DEVICE}")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation"

DATASETS = {
    "corsican": {
        "images": os.path.join(BASE, "corsican", "images"),
        "masks":  os.path.join(BASE, "corsican", "masks"),
    },
    "boreal": {
        "images": os.path.join(BASE, "boreal", "images"),
        "masks":  os.path.join(BASE, "boreal", "masks"),
    },
    "kaggle": {
        "images": os.path.join(BASE, "kaggle", "images"),
        "masks":  os.path.join(BASE, "kaggle", "masks"),
    },
}

print("Dataset paths:")
for name, paths in DATASETS.items():
    exists = os.path.isdir(paths["images"])
    print(f"  {name}: {'OK' if exists else 'NOT FOUND'} → {paths['images']}")

Dataset paths:
  corsican: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/corsican/images
  boreal: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/boreal/images
  kaggle: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/kaggle/images


Dataset

In [ ]:
class FireDataset(Dataset):
    def __init__(self, images_path, masks_path, transform=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.transform = transform
        self.images = sorted(os.listdir(images_path))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_path, img_name)

        stem = os.path.splitext(img_name)[0]
        mask_name = img_name.replace("_rgb", "_gt")
        mask_stem = os.path.splitext(mask_name)[0]

        for ext in [os.path.splitext(mask_name)[1], '.png', '.jpg', '.jpeg']:
            candidate = mask_stem + ext
            if os.path.exists(os.path.join(self.masks_path, candidate)):
                mask_name = candidate
                break

        mask_path = os.path.join(self.masks_path, mask_name)

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))
        mask  = (mask > 0).astype("float32")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"].unsqueeze(0)

        if image.dim() == 3 and (image.shape[1] != image.shape[2] or image.shape[1] == 0):
            target = image.shape[1]
            image = nn.functional.interpolate(
                image.unsqueeze(0), size=(target, target), mode='bilinear', align_corners=False
            ).squeeze(0)

        return image, mask


Data Augmentation

In [ ]:
DATASET_IMG_SIZE = {
    'corsican': 256,
    'boreal':   256,
    'kaggle':   256,
}

def get_transforms(img_size):
    """Return (train_transform, val_transform) for a given resolution."""
    train = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    val = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train, val

train_transform, val_transform = get_transforms(256)

Data Splits

In [ ]:
def create_data_splits(images_path, masks_path, train_transform, val_transform, data_percentage=1.0):
    """
    Create data splits with specific percentage of total dataset.

    Args:
        images_path: Path to images
        masks_path: Path to masks
        train_transform: Training transformations
        val_transform: Validation/test transformations
        data_percentage: Data split percentages (0.25, 0.5, 1.0)

    Returns:
        train_dataset, val_dataset, test_dataset
    """
    # Complete dataset
    total_size = len(FireDataset(images_path, masks_path))

    # Get size based on the percentage
    used_size = int(total_size * data_percentage)

    # Splits: 70% train, 15% val, 15% test
    train_size = int(0.7 * used_size)
    val_size = int(0.15 * used_size)
    test_size = used_size - train_size - val_size

    print(f"\nUsing {data_percentage*100:.0f}% of dataset:")
    print(f"  Total images: {total_size}")
    print(f"  Used images: {used_size}")
    print(f"  Train: {train_size}")
    print(f"  Val: {val_size}")
    print(f"  Test: {test_size}")

    # Create datasets
    full_train = FireDataset(images_path, masks_path, transform=train_transform)
    full_val = FireDataset(images_path, masks_path, transform=val_transform)
    full_test = FireDataset(images_path, masks_path, transform=val_transform)

    # Generar random indices
    indices = list(range(total_size))
    random.seed(42)
    random.shuffle(indices)

    # Take only the used percentage
    indices = indices[:used_size]

    # Divide in train/val/test
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create subsets
    train_dataset = Subset(full_train, train_indices)
    val_dataset = Subset(full_val, val_indices)
    test_dataset = Subset(full_test, test_indices)

    return train_dataset, val_dataset, test_dataset

Dice Loss (Loss function)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (
            preds.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

Metrics

In [ ]:
def get_metrics(preds, targets, threshold=0.5):
    """Calcula F1, mIoU, MCC, HAF"""
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    TP = (preds * targets).sum()
    TN = ((1 - preds) * (1 - targets)).sum()
    FP = (preds * (1 - targets)).sum()
    FN = ((1 - preds) * targets).sum()

    # F1 Score
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)

    # mIoU
    miou = TP / (TP + FP + FN + 1e-8)

    # MCC
    mcc = (TP * TN - FP * FN) / torch.sqrt(
        (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN) + 1e-8
    )

    # HAF
    haf = TP / (TP + FP + FN + 1e-8)

    return f1.item(), miou.item(), mcc.item(), haf.item()

Model

In [ ]:
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class WindowAttention(nn.Module):
    """Local window multi-head self-attention"""
    def __init__(self, dim, window_size=8, num_heads=4):
        super().__init__()
        self.ws = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.ws
        ph = (ws - H % ws) % ws
        pw = (ws - W % ws) % ws
        if ph or pw:
            x = nn.functional.pad(x, (0, pw, 0, ph))
        _, _, Hp, Wp = x.shape

        x = x.permute(0, 2, 3, 1)
        x = x.view(B, Hp//ws, ws, Wp//ws, ws, C).permute(0,1,3,2,4,5).contiguous()
        x = x.view(-1, ws*ws, C)

        qkv = self.qkv(x).reshape(-1, ws*ws, 3, self.num_heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(-1, ws*ws, C)
        out = self.proj(out)

        out = out.view(B, Hp//ws, Wp//ws, ws, ws, C).permute(0,1,3,2,4,5).contiguous()
        out = out.view(B, Hp, Wp, C).permute(0, 3, 1, 2)
        if ph or pw:
            out = out[:, :, :H, :W]
        return out


class STUBlock(nn.Module):
    """STU block: window-attention + conv MLP with residual."""
    def __init__(self, dim, window_size=8, num_heads=2):
        super().__init__()
        self.norm1 = nn.GroupNorm(1, dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.GroupNorm(1, dim)
        self.mlp = nn.Sequential(nn.Conv2d(dim, dim * 4, kernel_size=1), nn.GELU(), nn.Conv2d(dim * 4, dim, kernel_size=1))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class EncoderStage(nn.Module):
    """ConvBnRelu stem + N STUBlocks."""
    def __init__(self, in_ch, out_ch, num_blocks=2, window_size=8, num_heads=2):
        super().__init__()
        self.stem = ConvBnRelu(in_ch, out_ch, stride=2)
        self.blocks = nn.Sequential(*[STUBlock(out_ch, window_size, num_heads) for _ in range(num_blocks)])
    def forward(self, x): return self.blocks(self.stem(x))


class STUNetDenseFire(nn.Module):
    """STUNet + DenseFire for binary segmentation. dims=(32,64,128,256)."""
    def __init__(self, in_chans=3, dims=(32, 64, 128, 256), num_blocks=(2, 2, 2, 2), window_size=8):
        super().__init__()
        nh = (1, 2, 4, 8)
        self.enc1 = EncoderStage(in_chans, dims[0], num_blocks[0], window_size, nh[0])
        self.enc2 = EncoderStage(dims[0],  dims[1], num_blocks[1], window_size, nh[1])
        self.enc3 = EncoderStage(dims[1],  dims[2], num_blocks[2], window_size, nh[2])
        self.enc4 = EncoderStage(dims[2],  dims[3], num_blocks[3], window_size, nh[3])

        S = sum(dims) # 480
        self.dec3 = nn.Sequential(ConvBnRelu(dims[3]+S, dims[2]), ConvBnRelu(dims[2], dims[2]))
        self.dec2 = nn.Sequential(ConvBnRelu(dims[2]+S, dims[1]), ConvBnRelu(dims[1], dims[1]))
        self.dec1 = nn.Sequential(ConvBnRelu(dims[1]+S, dims[0]), ConvBnRelu(dims[0], dims[0]))
        self.dec0 = nn.Sequential(ConvBnRelu(dims[0]+S, dims[0]//2), ConvBnRelu(dims[0]//2, dims[0]//2))
        self.head = nn.Conv2d(dims[0] // 2, 1, kernel_size=1)

    def _dense_cat(self, d, enc_feats):
        H, W = d.shape[-2:]
        parts = [d]
        for f in enc_feats:
            if f.shape[-2:] != (H, W):
                f = nn.functional.interpolate(f, size=(H, W), mode='bilinear', align_corners=False)
            parts.append(f)
        return torch.cat(parts, dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        enc = [e1, e2, e3, e4]

        d = nn.functional.interpolate(e4, scale_factor=2, mode='bilinear', align_corners=False)
        d = self.dec3(self._dense_cat(d, enc))

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = self.dec2(self._dense_cat(d, enc))

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = self.dec1(self._dense_cat(d, enc))

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = self.dec0(self._dense_cat(d, enc))

        d = nn.functional.interpolate(d, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return self.head(d)


def build_stunet_densefire():
    return STUNetDenseFire(in_chans=3, dims=(32, 64, 128, 256), num_blocks=(2, 2, 2, 2), window_size=8)

_m = build_stunet_densefire()
_x = torch.zeros(2, 3, 256, 256)
_out = _m(_x)
print(f"STUNet+DenseFire output: {_out.shape}  params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m, _x, _out

STUNet+DenseFire output: torch.Size([2, 1, 256, 256])  params: 4.1M


Epoch Training and Validation

In [ ]:
scaler = GradScaler('cuda')

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    totals = dict(loss=0, f1=0, miou=0, mcc=0, haf=0)

    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        with autocast('cuda'):
            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            f1, miou, mcc, haf = get_metrics(logits, masks)
        totals['loss'] += loss.item()
        totals['f1'] += f1
        totals['miou'] += miou
        totals['mcc'] += mcc
        totals['haf']  += haf

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


def validate_epoch(model, loader, criterion):
    model.eval()
    totals = dict(loss=0, f1=0, miou=0, mcc=0, haf=0)

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, masks)
            f1, miou, mcc, haf = get_metrics(logits, masks)
            totals['loss'] += loss.item()
            totals['f1'] += f1
            totals['miou'] += miou
            totals['mcc'] += mcc
            totals['haf'] += haf

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


Experiment Configuration

In [ ]:
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 20
LR = 1e-4
PERCENTAGES = [0.25, 0.5, 1.0]

# Store results
all_histories = {}
all_test_results = {}

print("Starting training with different dataset percentages...\n")


Starting training with different dataset percentages...



Training Model

In [ ]:
def train_model(train_loader, val_loader, test_loader, dataset_name, data_percentage, num_epochs=NUM_EPOCHS, early_stopping_patience=EARLY_STOPPING_PATIENCE):
    global scaler
    scaler = GradScaler('cuda')

    pct_label = f"{int(data_percentage*100)}%"
    ckpt_name = f"stunet_densefire_{dataset_name}_{int(data_percentage*100)}pct.pth"

    print(f"\n{'='*70}")
    print(f"TRAINING WITH {data_percentage*100:.0f}% OF DATASET — {dataset_name.upper()}")
    print(f"{'='*70}\n")

    torch.manual_seed(SEED)
    model = build_stunet_densefire()
    model = model.to(DEVICE)

    criterion = DiceLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {'train_loss': [], 'train_f1': [], 'train_miou': [], 'train_mcc': [], 'train_haf': [], 'val_loss': [], 'val_f1': [], 'val_miou': [], 'val_mcc': [], 'val_haf': []}

    best_val_miou = 0.0
    best_model_path = ckpt_name
    patience_counter = 0

    for epoch in range(num_epochs):
        tr = train_epoch(model, train_loader, criterion, optimizer)
        vl = validate_epoch(model, val_loader, criterion)

        history['train_loss'].append(tr['loss'])
        history['train_f1'].append(tr['f1'])
        history['train_miou'].append(tr['miou'])
        history['train_mcc'].append(tr['mcc'])
        history['train_haf'].append(tr['haf'])

        history['val_loss'].append(vl['loss'])
        history['val_f1'].append(vl['f1'])
        history['val_miou'].append(vl['miou'])
        history['val_mcc'].append(vl['mcc'])
        history['val_haf'].append(vl['haf'])

        # Print progress every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train - Loss: {tr['loss']:.4f}, F1: {tr['f1']:.4f}, mIoU: {tr['miou']:.4f}")
            print(f"  Val   - Loss: {vl['loss']:.4f}, F1: {vl['f1']:.4f}, mIoU: {vl['miou']:.4f}")

        if vl['miou'] > best_val_miou:
            best_val_miou = vl['miou']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_miou': vl['miou'],
                'metrics': {'f1': vl['f1'], 'miou': vl['miou'], 'mcc': vl['mcc'], 'haf': vl['haf']}
            }, best_model_path)
            if (epoch + 1) % 5 == 0:
                print(f"  ✓ Best model saved! (mIoU: {best_val_miou:.4f})")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"  → Early stopping at epoch {epoch+1} "
                  f"(no improvement for {early_stopping_patience} epochs)")
            break

    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nBest model loaded from epoch {checkpoint['epoch']+1}")

    ts = validate_epoch(model, test_loader, criterion)

    print(f"\n{'='*70}")
    print(f"FINAL TEST RESULTS ({data_percentage*100:.0f}% dataset — {dataset_name.upper()})")
    print(f"{'='*70}")
    print(f"Loss:  {ts['loss']:.4f}")
    print(f"F1:    {ts['f1']:.4f}")
    print(f"mIoU:  {ts['miou']:.4f}")
    print(f"MCC:   {ts['mcc']:.4f}")
    print(f"HAF:   {ts['haf']:.4f}")
    print(f"{'='*70}\n")

    return history, {'test_loss': ts['loss'], 'test_f1': ts['f1'], 'test_miou': ts['miou'], 'test_mcc': ts['mcc'], 'test_haf': ts['haf'], 'epochs_run': epoch + 1}

Training — 3 datasets × 3 percentages

### Dataset: CORSICAN

In [ ]:
_img_size_corsican = DATASET_IMG_SIZE['corsican']
_tr_tf_corsican, _vl_tf_corsican = get_transforms(_img_size_corsican)
print(f"\nCORSICAN — resolution: {_img_size_corsican}x{_img_size_corsican}")

_, _val_ds_corsican, _test_ds_corsican = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=1.0
)
_val_loader_corsican = DataLoader(_val_ds_corsican,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_corsican = DataLoader(_test_ds_corsican, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



CORSICAN — resolution: 256x256

Using 100% of dataset:
  Total images: 500
  Used images: 500
  Train: 350
  Val: 75
  Test: 75


#### CORSICAN — 25% training data

In [ ]:
_train_ds_corsican_25, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=0.25
)
_train_loader_corsican_25 = DataLoader(
    _train_ds_corsican_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_25, _res_corsican_25 = train_model(
    _train_loader_corsican_25, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=0.25,
)

all_histories[('corsican', '25%')] = _hist_corsican_25
all_test_results[('corsican', '25%')] = _res_corsican_25



Using 25% of dataset:
  Total images: 500
  Used images: 125
  Train: 87
  Val: 18
  Test: 20

TRAINING WITH 25% OF DATASET — CORSICAN

Epoch [1/50]
  Train - Loss: 0.6361, F1: 0.4910, mIoU: 0.3379
  Val   - Loss: 0.6106, F1: 0.6264, mIoU: 0.4574
Epoch [5/50]
  Train - Loss: 0.5652, F1: 0.6817, mIoU: 0.5324
  Val   - Loss: 0.4693, F1: 0.7712, mIoU: 0.6291
  ✓ Best model saved! (mIoU: 0.6291)
Epoch [10/50]
  Train - Loss: 0.5371, F1: 0.7700, mIoU: 0.6319
  Val   - Loss: 0.4606, F1: 0.8108, mIoU: 0.6839
Epoch [15/50]
  Train - Loss: 0.5276, F1: 0.7860, mIoU: 0.6588
  Val   - Loss: 0.4465, F1: 0.8363, mIoU: 0.7209
  ✓ Best model saved! (mIoU: 0.7209)
Epoch [20/50]
  Train - Loss: 0.5157, F1: 0.8196, mIoU: 0.7015
  Val   - Loss: 0.4251, F1: 0.8476, mIoU: 0.7372
Epoch [25/50]
  Train - Loss: 0.5034, F1: 0.8351, mIoU: 0.7219
  Val   - Loss: 0.4249, F1: 0.8333, mIoU: 0.7163
Epoch [30/50]
  Train - Loss: 0.4990, F1: 0.8351, mIoU: 0.7246
  Val   - Loss: 0.4073, F1: 0.8488, mIoU: 0.7389
Epoch [

#### CORSICAN — 50% training data

In [ ]:
_train_ds_corsican_50, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=0.5
)
_train_loader_corsican_50 = DataLoader(
    _train_ds_corsican_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_50, _res_corsican_50 = train_model(
    _train_loader_corsican_50, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=0.5,
)

all_histories[('corsican', '50%')] = _hist_corsican_50
all_test_results[('corsican', '50%')] = _res_corsican_50



Using 50% of dataset:
  Total images: 500
  Used images: 250
  Train: 175
  Val: 37
  Test: 38

TRAINING WITH 50% OF DATASET — CORSICAN

Epoch [1/50]
  Train - Loss: 0.6311, F1: 0.5180, mIoU: 0.3581
  Val   - Loss: 0.5631, F1: 0.6691, mIoU: 0.5050
Epoch [5/50]
  Train - Loss: 0.5541, F1: 0.7231, mIoU: 0.5740
  Val   - Loss: 0.4509, F1: 0.8040, mIoU: 0.6738
  ✓ Best model saved! (mIoU: 0.6738)
Epoch [10/50]
  Train - Loss: 0.5292, F1: 0.7782, mIoU: 0.6463
  Val   - Loss: 0.4310, F1: 0.8450, mIoU: 0.7323
Epoch [15/50]
  Train - Loss: 0.5048, F1: 0.8041, mIoU: 0.6776
  Val   - Loss: 0.4184, F1: 0.8672, mIoU: 0.7657
Epoch [20/50]
  Train - Loss: 0.4909, F1: 0.8148, mIoU: 0.6929
  Val   - Loss: 0.4002, F1: 0.8770, mIoU: 0.7813
Epoch [25/50]
  Train - Loss: 0.4687, F1: 0.8426, mIoU: 0.7324
  Val   - Loss: 0.3758, F1: 0.8928, mIoU: 0.8065
  ✓ Best model saved! (mIoU: 0.8065)
Epoch [30/50]
  Train - Loss: 0.4460, F1: 0.8625, mIoU: 0.7626
  Val   - Loss: 0.3631, F1: 0.8745, mIoU: 0.7772
Epoch 

#### CORSICAN — 100% training data

In [ ]:
_train_ds_corsican_100, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=1.0
)
_train_loader_corsican_100 = DataLoader(
    _train_ds_corsican_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_100, _res_corsican_100 = train_model(
    _train_loader_corsican_100, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=1.0,
)

all_histories[('corsican', '100%')] = _hist_corsican_100
all_test_results[('corsican', '100%')] = _res_corsican_100



Using 100% of dataset:
  Total images: 500
  Used images: 500
  Train: 350
  Val: 75
  Test: 75

TRAINING WITH 100% OF DATASET — CORSICAN

Epoch [1/50]
  Train - Loss: 0.5918, F1: 0.6016, mIoU: 0.4384
  Val   - Loss: 0.4838, F1: 0.7436, mIoU: 0.5939
Epoch [5/50]
  Train - Loss: 0.5137, F1: 0.7564, mIoU: 0.6140
  Val   - Loss: 0.4345, F1: 0.8138, mIoU: 0.6870
Epoch [10/50]
  Train - Loss: 0.4751, F1: 0.8021, mIoU: 0.6747
  Val   - Loss: 0.3860, F1: 0.8683, mIoU: 0.7678
  ✓ Best model saved! (mIoU: 0.7678)
Epoch [15/50]
  Train - Loss: 0.4339, F1: 0.8406, mIoU: 0.7293
  Val   - Loss: 0.3585, F1: 0.8606, mIoU: 0.7555
Epoch [20/50]
  Train - Loss: 0.3965, F1: 0.8718, mIoU: 0.7752
  Val   - Loss: 0.3211, F1: 0.9042, mIoU: 0.8252
  ✓ Best model saved! (mIoU: 0.8252)
Epoch [25/50]
  Train - Loss: 0.3562, F1: 0.8877, mIoU: 0.8006
  Val   - Loss: 0.2846, F1: 0.9173, mIoU: 0.8474
  ✓ Best model saved! (mIoU: 0.8474)
Epoch [30/50]
  Train - Loss: 0.3153, F1: 0.9113, mIoU: 0.8380
  Val   - Loss: 

### Dataset: BOREAL

In [ ]:
_img_size_boreal = DATASET_IMG_SIZE['boreal']
_tr_tf_boreal, _vl_tf_boreal = get_transforms(_img_size_boreal)
print(f"\nBOREAL — resolution: {_img_size_boreal}x{_img_size_boreal}")

_, _val_ds_boreal, _test_ds_boreal = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=1.0
)
_val_loader_boreal = DataLoader(_val_ds_boreal,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_boreal = DataLoader(_test_ds_boreal, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



BOREAL — resolution: 256x256

Using 100% of dataset:
  Total images: 1417
  Used images: 1417
  Train: 991
  Val: 212
  Test: 214


#### BOREAL — 25% training data

In [ ]:
_train_ds_boreal_25, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=0.25
)
_train_loader_boreal_25 = DataLoader(
    _train_ds_boreal_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_25, _res_boreal_25 = train_model(
    _train_loader_boreal_25, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=0.25,
)

all_histories[   ('boreal', '25%')] = _hist_boreal_25
all_test_results[('boreal', '25%')] = _res_boreal_25



Using 25% of dataset:
  Total images: 1417
  Used images: 354
  Train: 247
  Val: 53
  Test: 54

TRAINING WITH 25% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.5736, F1: 0.5918, mIoU: 0.4362
  Val   - Loss: 0.5547, F1: 0.6107, mIoU: 0.4470
Epoch [5/50]
  Train - Loss: 0.4960, F1: 0.7046, mIoU: 0.5556
  Val   - Loss: 0.4918, F1: 0.7641, mIoU: 0.6256
  ✓ Best model saved! (mIoU: 0.6256)
Epoch [10/50]
  Train - Loss: 0.4565, F1: 0.7713, mIoU: 0.6370
  Val   - Loss: 0.4619, F1: 0.8031, mIoU: 0.6773
  ✓ Best model saved! (mIoU: 0.6773)
Epoch [15/50]
  Train - Loss: 0.4219, F1: 0.8219, mIoU: 0.7034
  Val   - Loss: 0.4264, F1: 0.7892, mIoU: 0.6590
Epoch [20/50]
  Train - Loss: 0.4012, F1: 0.8161, mIoU: 0.6966
  Val   - Loss: 0.4112, F1: 0.8074, mIoU: 0.6847
Epoch [25/50]
  Train - Loss: 0.3801, F1: 0.8390, mIoU: 0.7292
  Val   - Loss: 0.3732, F1: 0.8563, mIoU: 0.7521
  ✓ Best model saved! (mIoU: 0.7521)
Epoch [30/50]
  Train - Loss: 0.3508, F1: 0.8627, mIoU: 0.7649
  Val   - Loss: 0.3

#### BOREAL — 50% training data

In [ ]:
_train_ds_boreal_50, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=0.5
)
_train_loader_boreal_50 = DataLoader(
    _train_ds_boreal_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_50, _res_boreal_50 = train_model(
    _train_loader_boreal_50, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=0.5,
)

all_histories[('boreal', '50%')] = _hist_boreal_50
all_test_results[('boreal', '50%')] = _res_boreal_50



Using 50% of dataset:
  Total images: 1417
  Used images: 708
  Train: 495
  Val: 106
  Test: 107

TRAINING WITH 50% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.5565, F1: 0.6227, mIoU: 0.4647
  Val   - Loss: 0.5170, F1: 0.6894, mIoU: 0.5341
Epoch [5/50]
  Train - Loss: 0.4685, F1: 0.7327, mIoU: 0.5880
  Val   - Loss: 0.4673, F1: 0.8027, mIoU: 0.6754
  ✓ Best model saved! (mIoU: 0.6754)
Epoch [10/50]
  Train - Loss: 0.4171, F1: 0.7963, mIoU: 0.6738
  Val   - Loss: 0.3929, F1: 0.8319, mIoU: 0.7177
  ✓ Best model saved! (mIoU: 0.7177)
Epoch [15/50]
  Train - Loss: 0.3642, F1: 0.8478, mIoU: 0.7440
  Val   - Loss: 0.3521, F1: 0.8508, mIoU: 0.7440
  ✓ Best model saved! (mIoU: 0.7440)
Epoch [20/50]
  Train - Loss: 0.3135, F1: 0.8772, mIoU: 0.7848
  Val   - Loss: 0.2843, F1: 0.8846, mIoU: 0.7964
Epoch [25/50]
  Train - Loss: 0.2733, F1: 0.8868, mIoU: 0.7997
  Val   - Loss: 0.2568, F1: 0.8915, mIoU: 0.8070
Epoch [30/50]
  Train - Loss: 0.2236, F1: 0.9060, mIoU: 0.8314
  Val   - Loss: 0

#### BOREAL — 100% training data

In [ ]:
_train_ds_boreal_100, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=1.0
)
_train_loader_boreal_100 = DataLoader(
    _train_ds_boreal_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_100, _res_boreal_100 = train_model(
    _train_loader_boreal_100, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=1.0,
)

all_histories[('boreal', '100%')] = _hist_boreal_100
all_test_results[('boreal', '100%')] = _res_boreal_100



Using 100% of dataset:
  Total images: 1417
  Used images: 1417
  Train: 991
  Val: 212
  Test: 214

TRAINING WITH 100% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.5323, F1: 0.6492, mIoU: 0.4921
  Val   - Loss: 0.4730, F1: 0.6809, mIoU: 0.5238
Epoch [5/50]
  Train - Loss: 0.4177, F1: 0.6837, mIoU: 0.5302
  Val   - Loss: 0.3905, F1: 0.7038, mIoU: 0.5504
  ✓ Best model saved! (mIoU: 0.5504)
Epoch [10/50]
  Train - Loss: 0.3171, F1: 0.8719, mIoU: 0.7783
  Val   - Loss: 0.3133, F1: 0.8679, mIoU: 0.7693
Epoch [15/50]
  Train - Loss: 0.2302, F1: 0.9004, mIoU: 0.8225
  Val   - Loss: 0.2155, F1: 0.9082, mIoU: 0.8345
  ✓ Best model saved! (mIoU: 0.8345)
Epoch [20/50]
  Train - Loss: 0.1711, F1: 0.9106, mIoU: 0.8382
  Val   - Loss: 0.1650, F1: 0.9092, mIoU: 0.8360
  ✓ Best model saved! (mIoU: 0.8360)
Epoch [25/50]
  Train - Loss: 0.1366, F1: 0.9165, mIoU: 0.8483
  Val   - Loss: 0.1323, F1: 0.9093, mIoU: 0.8371
Epoch [30/50]
  Train - Loss: 0.1146, F1: 0.9195, mIoU: 0.8533
  Val   - Loss

### Dataset: KAGGLE

In [ ]:
_img_size_kaggle = DATASET_IMG_SIZE['kaggle']
_tr_tf_kaggle, _vl_tf_kaggle = get_transforms(_img_size_kaggle)
print(f"\nKAGGLE — resolution: {_img_size_kaggle}x{_img_size_kaggle}")

_, _val_ds_kaggle, _test_ds_kaggle = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=1.0
)
_val_loader_kaggle = DataLoader(_val_ds_kaggle,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_kaggle = DataLoader(_test_ds_kaggle, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



KAGGLE — resolution: 256x256

Using 100% of dataset:
  Total images: 27460
  Used images: 27460
  Train: 19222
  Val: 4119
  Test: 4119


#### KAGGLE — 25% training data

In [ ]:
_train_ds_kaggle_25, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=0.25
)
_train_loader_kaggle_25 = DataLoader(
    _train_ds_kaggle_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_25, _res_kaggle_25 = train_model(
    _train_loader_kaggle_25, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=0.25,
)

all_histories[('kaggle', '25%')] = _hist_kaggle_25
all_test_results[('kaggle', '25%')] = _res_kaggle_25



Using 25% of dataset:
  Total images: 27460
  Used images: 6865
  Train: 4805
  Val: 1029
  Test: 1031

TRAINING WITH 25% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.9094, F1: 0.1711, mIoU: 0.0942
  Val   - Loss: 0.8936, F1: 0.2091, mIoU: 0.1173
Epoch [5/50]
  Train - Loss: 0.6312, F1: 0.5190, mIoU: 0.3528
  Val   - Loss: 0.5986, F1: 0.5295, mIoU: 0.3623
  ✓ Best model saved! (mIoU: 0.3623)
Epoch [10/50]
  Train - Loss: 0.4013, F1: 0.6160, mIoU: 0.4476
  Val   - Loss: 0.3960, F1: 0.6179, mIoU: 0.4494
  ✓ Best model saved! (mIoU: 0.4494)
Epoch [15/50]
  Train - Loss: 0.3519, F1: 0.6513, mIoU: 0.4855
  Val   - Loss: 0.3580, F1: 0.6446, mIoU: 0.4776
  ✓ Best model saved! (mIoU: 0.4776)
Epoch [20/50]
  Train - Loss: 0.3285, F1: 0.6724, mIoU: 0.5087
  Val   - Loss: 0.3312, F1: 0.6695, mIoU: 0.5052
  ✓ Best model saved! (mIoU: 0.5052)
Epoch [25/50]
  Train - Loss: 0.3090, F1: 0.6914, mIoU: 0.5306
  Val   - Loss: 0.3300, F1: 0.6704, mIoU: 0.5061
  ✓ Best model saved! (mIoU: 0.5061)
E

#### KAGGLE — 50% training data

In [ ]:
_train_ds_kaggle_50, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=0.5
)
_train_loader_kaggle_50 = DataLoader(
    _train_ds_kaggle_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_50, _res_kaggle_50 = train_model(
    _train_loader_kaggle_50, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=0.5,
)

all_histories[('kaggle', '50%')] = _hist_kaggle_50
all_test_results[('kaggle', '50%')] = _res_kaggle_50



Using 50% of dataset:
  Total images: 27460
  Used images: 13730
  Train: 9611
  Val: 2059
  Test: 2060

TRAINING WITH 50% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.8908, F1: 0.2198, mIoU: 0.1254
  Val   - Loss: 0.8532, F1: 0.3651, mIoU: 0.2246
Epoch [5/50]
  Train - Loss: 0.4152, F1: 0.6034, mIoU: 0.4348
  Val   - Loss: 0.3922, F1: 0.6220, mIoU: 0.4536
  ✓ Best model saved! (mIoU: 0.4536)
Epoch [10/50]
  Train - Loss: 0.3427, F1: 0.6582, mIoU: 0.4930
  Val   - Loss: 0.3340, F1: 0.6669, mIoU: 0.5022
  ✓ Best model saved! (mIoU: 0.5022)
Epoch [15/50]
  Train - Loss: 0.3105, F1: 0.6898, mIoU: 0.5287
  Val   - Loss: 0.3138, F1: 0.6864, mIoU: 0.5242
Epoch [20/50]
  Train - Loss: 0.2871, F1: 0.7131, mIoU: 0.5563
  Val   - Loss: 0.2949, F1: 0.7053, mIoU: 0.5469
  ✓ Best model saved! (mIoU: 0.5469)
Epoch [25/50]
  Train - Loss: 0.2725, F1: 0.7277, mIoU: 0.5737
  Val   - Loss: 0.2984, F1: 0.7017, mIoU: 0.5429
Epoch [30/50]
  Train - Loss: 0.2571, F1: 0.7431, mIoU: 0.5929
  Val   - L

#### KAGGLE — 100% training data

In [ ]:
_train_ds_kaggle_100, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=1.0
)
_train_loader_kaggle_100 = DataLoader(
    _train_ds_kaggle_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_100, _res_kaggle_100 = train_model(
    _train_loader_kaggle_100, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=1.0,
)

all_histories[('kaggle', '100%')] = _hist_kaggle_100
all_test_results[('kaggle', '100%')] = _res_kaggle_100



Using 100% of dataset:
  Total images: 27460
  Used images: 27460
  Train: 19222
  Val: 4119
  Test: 4119

TRAINING WITH 100% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.8209, F1: 0.3294, mIoU: 0.2043
  Val   - Loss: 0.6558, F1: 0.5060, mIoU: 0.3407
Epoch [5/50]
  Train - Loss: 0.3505, F1: 0.6505, mIoU: 0.4845
  Val   - Loss: 0.3575, F1: 0.6430, mIoU: 0.4758
Epoch [10/50]
  Train - Loss: 0.2960, F1: 0.7041, mIoU: 0.5455
  Val   - Loss: 0.3034, F1: 0.6968, mIoU: 0.5365
Epoch [15/50]
  Train - Loss: 0.2668, F1: 0.7333, mIoU: 0.5808
  Val   - Loss: 0.2665, F1: 0.7336, mIoU: 0.5809
Epoch [20/50]
  Train - Loss: 0.2507, F1: 0.7494, mIoU: 0.6010
  Val   - Loss: 0.2558, F1: 0.7443, mIoU: 0.5946
Epoch [25/50]
  Train - Loss: 0.2390, F1: 0.7611, mIoU: 0.6159
  Val   - Loss: 0.2447, F1: 0.7555, mIoU: 0.6086
Epoch [30/50]
  Train - Loss: 0.2290, F1: 0.7711, mIoU: 0.6289
  Val   - Loss: 0.2387, F1: 0.7614, mIoU: 0.6163
Epoch [35/50]
  Train - Loss: 0.2212, F1: 0.7789, mIoU: 0.6392
  Val  